## About this Entry

Having the data we need at hand does not mean that we are ready to start analyzing it. Frequently, data is corrupted, inconsistent, or inadequate to our needs, which means that we must first solve these problems before we proceed, that is, we must first *clean* the data.

A dataset must meet certain criteria to be considered proper for use. When cleaning the data, we ensure that it is valid, accurate, complete, consistent, and uniform.

The first thing we must do is to import the packages with which we will work and load the data into a DataFrame.

In [406]:
import pandas as pd

In [407]:
df = pd.read_csv('datasets/retail_store_sales.csv')

Now, we examine the data to find possible inconsistencies or poor formating. To do this, the best methods are 1) printing part of the dataframe and 2) checking its `info()`, which tells us the data type of each column as well as how many null values they have

In [408]:
df

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02
...,...,...,...,...,...,...,...,...,...,...
12570,TXN_9347481,CUST_18,Patisserie,Item_23_PAT,38.0,4.0,152.0,Credit Card,In-store,2023-09-03
12571,TXN_4009414,CUST_03,Beverages,Item_2_BEV,6.5,9.0,58.5,Cash,Online,2022-08-12
12572,TXN_5306010,CUST_11,Butchers,Item_7_BUT,14.0,10.0,140.0,Cash,Online,2024-08-24
12573,TXN_5167298,CUST_04,Furniture,Item_7_FUR,14.0,6.0,84.0,Cash,Online,2023-12-30


In [409]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  object 
 1   Customer ID       12575 non-null  object 
 2   Category          12575 non-null  object 
 3   Item              11362 non-null  object 
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  object 
 8   Location          12575 non-null  object 
 9   Transaction Date  12575 non-null  object 
dtypes: float64(3), object(7)
memory usage: 982.6+ KB


By printing the DataFrame and checking its `info()` we can immediately spot some issues:
1. Columns such as 'Transaction ID', 'Customer ID', and 'Item' have unnecessary, redundant information in the form of prefixes and suffixes. Such information can be dropped in order to leave only the numeric data, which will allow us to change those columns data type from 'object' to 'int64', greatly reducing its size.
2. Columns such as 'Category', 'Payment Method', and 'Location' seem to be categorical. Therefore, changing their data type accordingly will improve pandas performance and reduce the size of the data.
3. The column 'Transaction Date' clearly contains datetime data, which means that we must also alter its type.
4. Finally, we have null values in multiple columns and must check if those values can be inputed or if they must be discarded.

### Changing Columns' Data Types

Because it is fairly easy to do so and the improvement in perfomance is significant, we should start our data cleaning process by attributing the correct data type to the columns we can. However, columns with null values may not be ready for this — NaNs are floats, which means that columns containing them may not be converted to other data types —, so they'll have to undergo further processing before we can adequate their data type.

Among the columns listed as needing data type conversion, 'Category', 'Payment Method', 'Location', and 'Transaction Date' don't have missing values. So let's start with them. Before we do so, however, it is important to know that categorical columns are characterized by having relatively few unique values, so we must check if this is true before changing their data type. 

In [410]:
df['Category'].unique()

array(['Patisserie', 'Milk Products', 'Butchers', 'Beverages', 'Food',
       'Furniture', 'Electric household essentials',
       'Computers and electric accessories'], dtype=object)

In [411]:
df['Payment Method'].unique()

array(['Digital Wallet', 'Credit Card', 'Cash'], dtype=object)

In [412]:
df['Location'].unique()

array(['Online', 'In-store'], dtype=object)

Having confirmed that they are, indeed, categorical, we can change their datatype.

In [413]:
df[['Category', 'Payment Method', 'Location']] = \
    df[['Category', 'Payment Method', 'Location']].astype('category')
df.dtypes

Transaction ID        object
Customer ID           object
Category            category
Item                  object
Price Per Unit       float64
Quantity             float64
Total Spent          float64
Payment Method      category
Location            category
Transaction Date      object
dtype: object

Now, for the 'Transaction Date' column:

In [414]:
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], format='%Y-%m-%d')
df.dtypes

Transaction ID              object
Customer ID                 object
Category                  category
Item                        object
Price Per Unit             float64
Quantity                   float64
Total Spent                float64
Payment Method            category
Location                  category
Transaction Date    datetime64[ns]
dtype: object

### Inputing/Discarding Null Values

Having converted the data type of the columns we could, now it is time to deal with the missing data.

Since we are trying to analyze which items sell best, we cannot work with data that contains no information on the item being sold. However, because we know that, for each category, there are no two items with the same price per unit, we can probably infer the missing items by their category and unit price. To do this, we can 1)separate the data by category, 2)create a dictionary with unit prices as keys and the corresponding items as values and 3)input the items by their unit price where they are missing.

Before we do so, however, it would be wise to first deal with the missing values in the 'Price Per Unit' column, since we need them to infer the items. This column, in its turn, is directly related to 'Quantity' and 'Total Spent', and we can calculate any missing value in any of the three columns as long as we have data on the other two. Where that is not the case, however, we'll unfortunately have to drop the data.

In [415]:
df = df.dropna(subset=['Price Per Unit', 'Quantity', 'Total Spent'], thresh=2)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11971 entries, 0 to 12574
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    11971 non-null  object        
 1   Customer ID       11971 non-null  object        
 2   Category          11971 non-null  category      
 3   Item              11362 non-null  object        
 4   Price Per Unit    11362 non-null  float64       
 5   Quantity          11971 non-null  float64       
 6   Total Spent       11971 non-null  float64       
 7   Payment Method    11971 non-null  category      
 8   Location          11971 non-null  category      
 9   Transaction Date  11971 non-null  datetime64[ns]
dtypes: category(3), datetime64[ns](1), float64(3), object(3)
memory usage: 783.9+ KB


Considering that the original dataframe had 12575 rows in it, 604 rows were dropped. Now, we can calculate the unit price for another 609 rows and input their items.

In [426]:
df.loc[:, 'Price Per Unit'] = df['Total Spent']/df['Quantity']

In [417]:
cat_dfs = {}

for cat in df['Category'].unique():
    cat_dfs[f'{cat.lower()}_df'] = df[df['Category'] == cat]

cat_dfs.keys()

dict_keys(['patisserie_df', 'milk products_df', 'butchers_df', 'beverages_df', 'food_df', 'furniture_df', 'electric household essentials_df', 'computers and electric accessories_df'])

Now that we have a dataframe for each category of products, we can sort them by 'Price Per Unit' and 'Item' and then forward fill the NaN values. This will work because, by ordering the dataframes by 'Price Per Unit' and 'Item', the NaN items will be placed directly below their labels, since they have the same price. By then using the `.fillna()` method with the `ffill` option, the labels will be propagated downwards when a NaN value is encountered. Take a look at the following example:

In [418]:
cat_dfs['furniture_df'].drop_duplicates(subset=['Item', 'Price Per Unit']).sort_values(['Price Per Unit', 'Item']).head(13)

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date
140,TXN_4801202,CUST_16,Furniture,Item_1_FUR,5.0,3.0,15.0,Digital Wallet,Online,2024-05-31
700,TXN_3010762,CUST_24,Furniture,NaN,5.0,2.0,10.0,Digital Wallet,In-store,2023-07-14
149,TXN_3541663,CUST_25,Furniture,Item_2_FUR,6.5,5.0,32.5,Digital Wallet,In-store,2023-07-19
2007,TXN_2292675,CUST_22,Furniture,NaN,6.5,1.0,6.5,Digital Wallet,Online,2024-06-23
757,TXN_5381793,CUST_05,Furniture,Item_3_FUR,8.0,9.0,72.0,Cash,In-store,2024-12-09
360,TXN_9659532,CUST_21,Furniture,Item_4_FUR,9.5,8.0,76.0,Cash,In-store,2023-01-08
6697,TXN_1211345,CUST_19,Furniture,NaN,9.5,7.0,66.5,Digital Wallet,Online,2024-12-02
108,TXN_9279462,CUST_25,Furniture,Item_5_FUR,11.0,3.0,33.0,Credit Card,In-store,2022-02-05
885,TXN_3512388,CUST_14,Furniture,NaN,11.0,10.0,110.0,Digital Wallet,In-store,2024-03-30
466,TXN_4882798,CUST_23,Furniture,Item_6_FUR,12.5,10.0,125.0,Credit Card,In-store,2022-04-22


In [419]:
for cat in cat_dfs.keys():
    cat_dfs[cat] = cat_dfs[cat].sort_values(['Price Per Unit', 'Item'])
    cat_dfs[cat].loc[:, 'Item'] = cat_dfs[cat].loc[:, 'Item'].ffill()

Again, let's take a look at the same dataframe to see that the NaN values have been filled:

In [420]:
cat_dfs['furniture_df'].drop_duplicates(subset=['Item', 'Price Per Unit']).sort_values(['Price Per Unit', 'Item']).head(7)

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date
140,TXN_4801202,CUST_16,Furniture,Item_1_FUR,5.0,3.0,15.0,Digital Wallet,Online,2024-05-31
149,TXN_3541663,CUST_25,Furniture,Item_2_FUR,6.5,5.0,32.5,Digital Wallet,In-store,2023-07-19
757,TXN_5381793,CUST_05,Furniture,Item_3_FUR,8.0,9.0,72.0,Cash,In-store,2024-12-09
360,TXN_9659532,CUST_21,Furniture,Item_4_FUR,9.5,8.0,76.0,Cash,In-store,2023-01-08
108,TXN_9279462,CUST_25,Furniture,Item_5_FUR,11.0,3.0,33.0,Credit Card,In-store,2022-02-05
466,TXN_4882798,CUST_23,Furniture,Item_6_FUR,12.5,10.0,125.0,Credit Card,In-store,2022-04-22
55,TXN_2383377,CUST_15,Furniture,Item_7_FUR,14.0,3.0,42.0,Digital Wallet,In-store,2023-04-02


All seems fine. But we need to be sure.

In [421]:
for cat in cat_dfs.keys():
    assert cat_dfs[cat]['Item'].notnull().all()

Since the assertion does not raise an error, we know that there aren't any more NA values in the 'Items' column, so we can concatenate the separate dataframes back into a single one.

In [422]:
df = pd.concat(cat_dfs).reset_index(drop=True)
df

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date
0,TXN_2016332,CUST_17,Patisserie,Item_1_PAT,5.0,6.0,30.0,Digital Wallet,Online,2022-12-17
1,TXN_6590263,CUST_14,Patisserie,Item_1_PAT,5.0,7.0,35.0,Digital Wallet,Online,2024-12-11
2,TXN_5314835,CUST_22,Patisserie,Item_1_PAT,5.0,7.0,35.0,Digital Wallet,Online,2024-07-05
3,TXN_2533200,CUST_05,Patisserie,Item_1_PAT,5.0,8.0,40.0,Credit Card,Online,2022-10-28
4,TXN_8785418,CUST_07,Patisserie,Item_1_PAT,5.0,9.0,45.0,Credit Card,Online,2023-02-12
...,...,...,...,...,...,...,...,...,...,...
11966,TXN_5870737,CUST_17,Computers and electric accessories,Item_25_CEA,41.0,10.0,410.0,Credit Card,Online,2024-12-24
11967,TXN_6970525,CUST_06,Computers and electric accessories,Item_25_CEA,41.0,3.0,123.0,Credit Card,In-store,2023-01-27
11968,TXN_6964148,CUST_09,Computers and electric accessories,Item_25_CEA,41.0,1.0,41.0,Cash,Online,2024-03-15
11969,TXN_3855536,CUST_04,Computers and electric accessories,Item_25_CEA,41.0,10.0,410.0,Credit Card,In-store,2024-09-27


In [423]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11971 entries, 0 to 11970
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    11971 non-null  object        
 1   Customer ID       11971 non-null  object        
 2   Category          11971 non-null  category      
 3   Item              11971 non-null  object        
 4   Price Per Unit    11971 non-null  float64       
 5   Quantity          11971 non-null  float64       
 6   Total Spent       11971 non-null  float64       
 7   Payment Method    11971 non-null  category      
 8   Location          11971 non-null  category      
 9   Transaction Date  11971 non-null  datetime64[ns]
dtypes: category(3), datetime64[ns](1), float64(3), object(3)
memory usage: 690.5+ KB


### Dealing with Unnecessary Info

There are no NA values left in our dataframe. Great! Now, we can focus on cleaning the columns 'Transaction ID', 'Customer ID', and 'Item' from unnecessary information and chenge their data types. This will significantly reduce data size and improve Pandas' performance when dealing with it — if you haven't noticed, we have already reduced the dataset's memory usage by almost a third. Take a look at the first call to `info()` and this last one and you will see that the size has gone from 982.6+ KB to 690.5+ KB! That may not seem like much when considering a single file, but imagine the impact of doing this to all of an organization's data.

Let's extract the useful information:

In [427]:
df.loc[:, 'Transaction ID'] = df['Transaction ID'].replace('^TXN_', '', regex=True)
df.loc[:, 'Customer ID'] = df['Customer ID'].replace('^CUST_', '', regex=True)
df.loc[:, 'Item'] = df['Item'].str.extract(r'^Item_(\d+)_')[0]
df

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date
0,2016332,17,Patisserie,1,5.0,6.0,30.0,Digital Wallet,Online,2022-12-17
1,6590263,14,Patisserie,1,5.0,7.0,35.0,Digital Wallet,Online,2024-12-11
2,5314835,22,Patisserie,1,5.0,7.0,35.0,Digital Wallet,Online,2024-07-05
3,2533200,05,Patisserie,1,5.0,8.0,40.0,Credit Card,Online,2022-10-28
4,8785418,07,Patisserie,1,5.0,9.0,45.0,Credit Card,Online,2023-02-12
...,...,...,...,...,...,...,...,...,...,...
11966,5870737,17,Computers and electric accessories,25,41.0,10.0,410.0,Credit Card,Online,2024-12-24
11967,6970525,06,Computers and electric accessories,25,41.0,3.0,123.0,Credit Card,In-store,2023-01-27
11968,6964148,09,Computers and electric accessories,25,41.0,1.0,41.0,Cash,Online,2024-03-15
11969,3855536,04,Computers and electric accessories,25,41.0,10.0,410.0,Credit Card,In-store,2024-09-27


Nice! Now we can change the columns' datatype to 'int', which takes significantly less space and performs significantly better than 'object' columns. I will also change 'Quantity' dtype to 'int', since we cannot sell fractions of items and since we have dealt with the NA values.

In [428]:
df[['Transaction ID', 'Customer ID', 'Item', 'Quantity']] = \
    df[['Transaction ID', 'Customer ID', 'Item', 'Quantity']].astype('int')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11971 entries, 0 to 11970
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    11971 non-null  int64         
 1   Customer ID       11971 non-null  int64         
 2   Category          11971 non-null  category      
 3   Item              11971 non-null  int64         
 4   Price Per Unit    11971 non-null  float64       
 5   Quantity          11971 non-null  int64         
 6   Total Spent       11971 non-null  float64       
 7   Payment Method    11971 non-null  category      
 8   Location          11971 non-null  category      
 9   Transaction Date  11971 non-null  datetime64[ns]
dtypes: category(3), datetime64[ns](1), float64(2), int64(4)
memory usage: 690.5 KB


### One Last Thing

Our data is now (almost) perfectly ready for being analyzed. There is just one problem: if we take a look only at the 'Item' column, we are unable to tell one item from another, since now that we have stripped them from unnecessary and redundant information, they are not uniquely identified anymore. That is because, for each item category, we have 25 different items, ranging from 1 to 25. Between categories, however, the numbers repeat. We must avoid this.

In [436]:
items_df = df[['Category', 'Item']].drop_duplicates(subset=['Category', 'Item'])
items_df

,Category,Item
0,Patisserie,1
68,Patisserie,2
108,Patisserie,3
135,Patisserie,4
212,Patisserie,5
...,...,...
11759,Computers and electric accessories,21
11818,Computers and electric accessories,22
11872,Computers and electric accessories,23
11920,Computers and electric accessories,24


In [438]:
items_df['new_id'] = range(1, len(items_df)+1)
items_df

,Category,Item,new_id
0,Patisserie,1,1
68,Patisserie,2,2
108,Patisserie,3,3
135,Patisserie,4,4
212,Patisserie,5,5
...,...,...,...
11759,Computers and electric accessories,21,196
11818,Computers and electric accessories,22,197
11872,Computers and electric accessories,23,198
11920,Computers and electric accessories,24,199


In [439]:
df = df.merge(items_df, on=['Category', 'Item'], how='left')
df

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,new_id
0,2016332,17,Patisserie,1,5.0,6,30.0,Digital Wallet,Online,2022-12-17,1
1,6590263,14,Patisserie,1,5.0,7,35.0,Digital Wallet,Online,2024-12-11,1
2,5314835,22,Patisserie,1,5.0,7,35.0,Digital Wallet,Online,2024-07-05,1
3,2533200,5,Patisserie,1,5.0,8,40.0,Credit Card,Online,2022-10-28,1
4,8785418,7,Patisserie,1,5.0,9,45.0,Credit Card,Online,2023-02-12,1
...,...,...,...,...,...,...,...,...,...,...,...
11966,5870737,17,Computers and electric accessories,25,41.0,10,410.0,Credit Card,Online,2024-12-24,200
11967,6970525,6,Computers and electric accessories,25,41.0,3,123.0,Credit Card,In-store,2023-01-27,200
11968,6964148,9,Computers and electric accessories,25,41.0,1,41.0,Cash,Online,2024-03-15,200
11969,3855536,4,Computers and electric accessories,25,41.0,10,410.0,Credit Card,In-store,2024-09-27,200


In [441]:
df['Item'] = df['new_id']
df = df.drop('new_id', axis=1)
df

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date
0,2016332,17,Patisserie,1,5.0,6,30.0,Digital Wallet,Online,2022-12-17
1,6590263,14,Patisserie,1,5.0,7,35.0,Digital Wallet,Online,2024-12-11
2,5314835,22,Patisserie,1,5.0,7,35.0,Digital Wallet,Online,2024-07-05
3,2533200,5,Patisserie,1,5.0,8,40.0,Credit Card,Online,2022-10-28
4,8785418,7,Patisserie,1,5.0,9,45.0,Credit Card,Online,2023-02-12
...,...,...,...,...,...,...,...,...,...,...
11966,5870737,17,Computers and electric accessories,200,41.0,10,410.0,Credit Card,Online,2024-12-24
11967,6970525,6,Computers and electric accessories,200,41.0,3,123.0,Credit Card,In-store,2023-01-27
11968,6964148,9,Computers and electric accessories,200,41.0,1,41.0,Cash,Online,2024-03-15
11969,3855536,4,Computers and electric accessories,200,41.0,10,410.0,Credit Card,In-store,2024-09-27


In [442]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11971 entries, 0 to 11970
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    11971 non-null  int64         
 1   Customer ID       11971 non-null  int64         
 2   Category          11971 non-null  category      
 3   Item              11971 non-null  int64         
 4   Price Per Unit    11971 non-null  float64       
 5   Quantity          11971 non-null  int64         
 6   Total Spent       11971 non-null  float64       
 7   Payment Method    11971 non-null  category      
 8   Location          11971 non-null  category      
 9   Transaction Date  11971 non-null  datetime64[ns]
dtypes: category(3), datetime64[ns](1), float64(2), int64(4)
memory usage: 690.5 KB
